In [1]:
import os
import operator
from datetime import datetime
from typing import Annotated
from typing_extensions import TypedDict

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

In [2]:
load_dotenv()

GroqApiKey = os.getenv("GroqAPI")
if not GroqApiKey:
    raise ValueError("GroqAPI key not found in .env")

LLM = ChatGroq(
    api_key = GroqApiKey,
    model = "llama-3.3-70b-versatile",
    temperature = 0.3,
)

In [3]:
class DocumentState(TypedDict):
    RawText: str
    Summary: str
    Topics: str
    Sentiment: str
    Logs: Annotated[list, operator.add]

In [4]:
def makeTimestamp():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def callLlm(Prompt: str) -> str:
    Response = LLM.invoke(Prompt)
    return Response.content.strip()

def loadDocument(State: DocumentState) -> dict:
    FilePath = State["RawText"]

    print(f"[{makeTimestamp()}] Node: Load Document - reading '{FilePath}'")

    with open(FilePath, "r", encoding="utf-8") as File:
        FileContent = File.read()

    LogEntry = f"[{makeTimestamp()}] Load Document: Loaded {len(FileContent)} characters from '{FilePath}'"
    print(LogEntry)

    return { "RawText": FileContent, "Logs": [LogEntry], }

def summariseDocument(State: DocumentState) -> dict:
    
    print(f"[{makeTimestamp()}] Node: summarise - calling LLM")

    Prompt = (
        f"You are a document analyst. Read the following document and write a concise summary of 3 to 5 sentences. Only return the summary.\n\n Document:\n{State['RawText']}"
    )

    Result = callLlm(Prompt)

    LogEntry = f"[{makeTimestamp()}] summarise: completed ({len(Result)} characters returned)"
    print(LogEntry)

    return {"Summary": Result, "Logs": [LogEntry], }


In [5]:
def extractTopics(State: DocumentState) -> dict:

    print(f"[{makeTimestamp()}] Node: Extract Topics - calling LLM")

    Prompt = (
        f"You are a document analyst. Read the following document and extract the 5 most important topics or themes. Return them as a numbered list. Only return the list.\n\n Document:\n{State['RawText']}"
    )

    Result = callLlm(Prompt)

    LogEntry = f"[{makeTimestamp()}] Extract Topics: Completed ({len(Result)} characters returned)"
    print(LogEntry)

    return { "Topics": Result, "Logs": [LogEntry], }

def analyseSentiment(State: DocumentState) -> dict:

    print(f"[{makeTimestamp()}] Node: Analyse Sentiment - calling LLM")

    Prompt = (
        f"You are a sentiment analysis expert. Read the following document: 1. State the overall sentiment (Positive / Negative / Neutral / Mixed) 2. Give a confidence score from 0 to 100 3. Write 2 to 3 sentences explaining your reasoning. \nOnly return these three items.\n\n Document:\n{State['RawText']}"
    )

    Result = callLlm(Prompt)

    LogEntry = f"[{makeTimestamp()}] Analyse Sentiment: Completed ({len(Result)} characters returned)"
    print(LogEntry)

    return { "Sentiment": Result, "Logs": [LogEntry], }

def mergeResults(State: DocumentState) -> dict:

    print(f"[{makeTimestamp()}] Node: Merge Results - Assembling final report")

    Report = (
        f"\n\n"
        
        f"Document Analysis Report\n"
        f"Generated: {makeTimestamp()}\n"
        
        f"\n\n"
        
        f"Summary:\n\n{State['Summary']}\n\n"
        f"Key Topics:\n\n{State['Topics']}\n\n"
        f"Sentiment Analysis\n\n{State['Sentiment']}\n\n"
        
        f"\n\n"
    )

    print(Report)

    LogEntry = f"[{makeTimestamp()}] Merge Results: Final report assembled successfully"
    print(LogEntry)

    return {"Logs": [LogEntry]}

In [6]:
GraphBuilder = StateGraph(DocumentState)

GraphBuilder.add_node("load_document", loadDocument)
GraphBuilder.add_node("summarise", summariseDocument)
GraphBuilder.add_node("extract_topics", extractTopics)
GraphBuilder.add_node("analyse_sentiment", analyseSentiment)
GraphBuilder.add_node("merge_results", mergeResults)

GraphBuilder.add_edge(START, "load_document")

GraphBuilder.add_edge("load_document", "summarise")
GraphBuilder.add_edge("load_document", "extract_topics")
GraphBuilder.add_edge("load_document", "analyse_sentiment")

GraphBuilder.add_edge("summarise", "merge_results")
GraphBuilder.add_edge("extract_topics", "merge_results")
GraphBuilder.add_edge("analyse_sentiment", "merge_results")

GraphBuilder.add_edge("merge_results", END)

App = GraphBuilder.compile()

In [7]:
print(App.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	load_document(load_document)
	summarise(summarise)
	extract_topics(extract_topics)
	analyse_sentiment(analyse_sentiment)
	merge_results(merge_results)
	__end__([<p>__end__</p>]):::last
	__start__ --> load_document;
	analyse_sentiment --> merge_results;
	extract_topics --> merge_results;
	load_document --> analyse_sentiment;
	load_document --> extract_topics;
	load_document --> summarise;
	summarise --> merge_results;
	merge_results --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [8]:
InputFilePath = "sample_input.txt"

InitialState = {
    "RawText": InputFilePath,
    "Summary": "",
    "Topics": "",
    "Sentiment": "",
    "Logs": [],
}

print(f"[{makeTimestamp()}] Starting graph execution\n")

FinalState = App.invoke(InitialState)

print(f"\n[{makeTimestamp()}] Graph execution complete")

[2026-06-10 16:28:55] Starting graph execution

[2026-06-10 16:28:55] Node: Load Document - reading 'sample_input.txt'
[2026-06-10 16:28:55] Load Document: Loaded 1788 characters from 'sample_input.txt'
[2026-06-10 16:28:55] Node: Analyse Sentiment - calling LLM
[2026-06-10 16:28:55] Node: Extract Topics - calling LLM
[2026-06-10 16:28:55] Node: summarise - calling LLM


APIConnectionError: Connection error.

In [ ]:
def saveReport(State: DocumentState, InputPath: str):
 
    Timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    OutputFileName = f"report_{Timestamp}.txt"

    ReportSection = (
        f"\n\n"
        
        f"Document Analysis Report\n"
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
        f"Source file: {InputPath}\n"
        
        f"\n\n"
        
        f"Summary\n\n{State['Summary']}\n\n"
        f"Key Topics\n\n{State['Topics']}\n\n"
        f"Sentiment Analysis\n\n{State['Sentiment']}\n\n"
        
        f"\n\n"
    )

    TraceSection = (
        f"\n\n"
        f"Execution Trace\n"
        f"\n\n"
    )

    for LogLine in State["Logs"]:
        TraceSection += LogLine + "\n"

    with open(OutputFileName, "w", encoding="utf-8") as OutputFile:
        OutputFile.write(ReportSection)
        OutputFile.write(TraceSection)

    print(f"Report saved to: {OutputFileName}")
    return OutputFileName


SavedFile = saveReport(FinalState, InputFilePath)